In [10]:
from datasets import load_dataset

ds = load_dataset("westenfelder/NL2SH-ALFA", "train")

def create_conversation(sample):
  return {
      "messages": [
          {"role": "system", "content": "You are a helpful assistant that translates natural language to bash commands."},
          {"role": "user", "content": "Generate single Bash command: " + sample["nl"]},
          {"role": "assistant", "content": f'```bash\n{sample["bash"]}\n```'},
      ]
  }

train_dataset = ds['train'].map(create_conversation, remove_columns=ds['train'].features, batched=False)

dataset = train_dataset.train_test_split(test_size=0.2, shuffle=True)


Map: 100%|██████████| 40639/40639 [00:00<00:00, 43513.13 examples/s]


In [11]:
print(dataset["train"][0]["messages"])

[{'content': 'You are a helpful assistant that translates natural language to bash commands.', 'role': 'system'}, {'content': 'Generate single Bash command: Determine the MTU to the destination', 'role': 'user'}, {'content': '```bash\ntraceroute --mtu example.com\n```', 'role': 'assistant'}]


In [12]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Load model and tokenizer
model = AutoModelForCausalLM.from_pretrained(
    "google/gemma-3-270m-it",
    dtype="auto",
    device_map="cuda",
    attn_implementation="eager"
)
tokenizer = AutoTokenizer.from_pretrained("google/gemma-3-270m-it")

print(f"Device: {model.device}")
print(f"DType: {model.dtype}")

Device: cuda:0
DType: torch.bfloat16


In [4]:
from transformers import pipeline

from random import randint
import re

# Load the model and tokenizer into the pipeline
pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)

# Load a random sample from the test dataset
rand_idx = randint(0, len(dataset["test"])-1)
test_sample = dataset["test"][rand_idx]

# Convert as test example into a prompt with the Gemma template
prompt = pipe.tokenizer.apply_chat_template(test_sample["messages"][:1], tokenize=False, add_generation_prompt=True)
outputs = pipe(prompt, max_new_tokens=256, disable_compile=True)

# Extract the user query and original answer
print(f"Question:\n{test_sample['messages'][0]['content']}\n")
print(f"Original Answer:\n{test_sample['messages'][1]['content']}\n")
print(f"Generated Answer (base model):\n{outputs[0]['generated_text'][len(prompt):].strip()}")

Device set to use cuda


Question:
Generate single Bash command: Find all files in the current directory, excluding those beginning with "#", list their details in long format, and sort them in reverse order by their fourth field.

Original Answer:
grep -vE "^#" <(find $(echo * -maxdepth 0) -type f) | xargs ls -l | sort -nt,2 -k4 -r

Generated Answer (base model):
```bash
find. -maxdepth 1 -type f -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -


In [14]:
message = [
    {"role": "system", "content": "You are a helpful assistant that translates natural language to bash commands."}
]

message.append(
    {"role": "user", "content": test_sample["messages"][:1][0]['content']}
)
outputs = pipe(message, max_new_tokens=256, disable_compile=True)
print(outputs[0]['generated_text'])
print("-"*80)
print(outputs[0]['generated_text'][-1]['content'])

[{'role': 'system', 'content': 'You are a helpful assistant that translates natural language to bash commands.'}, {'role': 'user', 'content': 'Generate single Bash command: Find all files in the current directory, excluding those beginning with "#", list their details in long format, and sort them in reverse order by their fourth field.'}]
[{'role': 'system', 'content': 'You are a helpful assistant that translates natural language to bash commands.'}, {'role': 'user', 'content': 'Generate single Bash command: Find all files in the current directory, excluding those beginning with "#", list their details in long format, and sort them in reverse order by their fourth field.'}, {'role': 'assistant', 'content': '```bash\nfind. -type f -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not -not

In [ ]:
from trl import SFTConfig
learning_rate = 5e-5
torch_dtype = model.dtype

args = SFTConfig(
    output_dir=f"gemma-3-270m-it-ft-bash",              # directory to save and repository id
    max_length=512,                         # max sequence length for model and packing of the dataset
    packing=False,                          # Groups multiple samples in the dataset into a single sequence
    num_train_epochs=5,                     # number of training epochs
    per_device_train_batch_size=4,          # batch size per device during training
    gradient_checkpointing=False,           # Caching is incompatible with gradient checkpointing
    optim="adamw_torch_fused",              # use fused adamw optimizer
    logging_steps=1,                        # log every step
    save_strategy="epoch",                  # save checkpoint every epoch
    eval_strategy="epoch",                  # evaluate checkpoint every epoch
    learning_rate=learning_rate,            # learning rate
    fp16=True if torch_dtype == torch.float16 else False,   # use float16 precision
    bf16=True if torch_dtype == torch.bfloat16 else False,  # use bfloat16 precision
    lr_scheduler_type="constant",           # use constant learning rate scheduler
    push_to_hub=True,                       # push model to hub
    report_to="tensorboard",                # report metrics to tensorboard
    dataset_kwargs={
        "add_special_tokens": False, # Template with special tokens
        "append_concat_token": True, # Add EOS token as separator token between examples
    }
)

In [18]:
from trl import SFTTrainer

# Create Trainer object
trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=dataset['train'],
    eval_dataset=dataset['test'],
    processing_class=tokenizer,
)

In [19]:
# Start training, the model will be automatically saved to the Hub and the output directory
trainer.train()

# Save the final model again to the Hugging Face Hub
trainer.save_model()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 2, 'pad_token_id': 0}.


Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
1,0.992600,0.938504,0.918691,2188280.000000,0.803903
2,0.672600,0.898868,0.752329,4376560.000000,0.813216
3,0.912200,0.913020,0.661497,6564840.000000,0.816186
4,0.565200,0.956957,0.600286,8753120.000000,0.816274
5,0.433200,1.032456,0.535544,10941400.000000,0.814104


In [20]:
import matplotlib.pyplot as plt

# Access the log history
log_history = trainer.state.log_history

# Extract training / validation loss
train_losses = [log["loss"] for log in log_history if "loss" in log]
epoch_train = [log["epoch"] for log in log_history if "loss" in log]
eval_losses = [log["eval_loss"] for log in log_history if "eval_loss" in log]
epoch_eval = [log["epoch"] for log in log_history if "eval_loss" in log]

# Plot the training loss
plt.plot(epoch_train, train_losses, label="Training Loss")
plt.plot(epoch_eval, eval_losses, label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss per Epoch")
plt.legend()
plt.grid(True)
plt.show()

ModuleNotFoundError: No module named 'matplotlib'

In [21]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "./checkpoints"

# Load Model
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype="auto",
    device_map="auto",
    attn_implementation="eager"
)
tokenizer = AutoTokenizer.from_pretrained(model_id)

`torch_dtype` is deprecated! Use `dtype` instead!


In [30]:
from transformers import pipeline

prompt = "copy all .jpg files to /mnt/backup"
model_input = {
    "messages": [
          {"role": "system", "content": "You are a helpful assistant that translates natural language to bash commands."},
          {"role": "user", "content": f"Generate single Bash command: {prompt}"},
      ]
}

output = pipe(model_input["messages"], max_new_tokens=256, disable_compile=True, clean_up_tokenization_spaces=False)[0]

generator = pipeline("text-generation", model="micrictor/checkpoints", device="cpu")
output = generator(model_input["messages"],
    max_new_tokens=256,
    disable_compile=True,
    clean_up_tokenization_spaces=False,
    return_full_text=False,
)[0]
print(output["generated_text"])

Device set to use cpu


```bash
find . -name '*.jpg' -exec cp "{}" "/mnt/backup/" \;
```
